# Qwen3-8B Chatbot — LoRA Fine-Tuning (Colab H100)

Colab'de H100 GPU seçip Runtime → Run all çalıştır. Qwen3-8B instruct modelini UltraChat sohbet dataset'iyle BF16 + LoRA SFT yapar. Çıktı full model değil, adapter'dır. HF model erişimi/lisans şartlarını kabul et. UltraChat genel sohbet verisidir, Türkçe chatbot kalitesi için sonradan özel Türkçe verilerle iyileştirme gerekir.


In [ ]:
!pip -q install -U "transformers>=4.51.0" "trl>=0.18.0" "peft>=0.15.0" "datasets>=3.6.0" "accelerate>=1.6.0" bitsandbytes huggingface_hub


In [ ]:
import torch, transformers, trl, peft
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU NOT FOUND')
print('CUDA:', torch.version.cuda, '| transformers:', transformers.__version__, '| TRL:', trl.__version__)
assert torch.cuda.is_available(), 'Colab GPU runtime seç.'
assert torch.cuda.is_bf16_supported(), 'BF16 destekleyen GPU (H100 önerilir) gerekiyor.'


## Hugging Face authentication (optional)
HF_TOKEN'ı Colab Secrets içine HF_TOKEN adıyla kaydet; token'ı notebook'a düz metin olarak yazma.

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token: login(token=token)
    else: print('HF_TOKEN bulunamadı; public erişim deneniyor.')
except Exception as e: print('HF login atlandı:', str(e)[:200])


## Dataset download
UltraChat 200k train_sft ve test_sft split'lerinden sınırlı örnek alınır. Türkçe oranı garanti değildir.

In [ ]:
from datasets import load_dataset
DATASET_ID = 'HuggingFaceH4/ultrachat_200k'
MAX_TRAIN_SAMPLES, MAX_EVAL_SAMPLES = 10000, 500
train_raw = load_dataset(DATASET_ID, split='train_sft').shuffle(seed=42)
train_raw = train_raw.select(range(min(MAX_TRAIN_SAMPLES, len(train_raw))))
eval_raw = load_dataset(DATASET_ID, split='test_sft').shuffle(seed=42)
eval_raw = eval_raw.select(range(min(MAX_EVAL_SAMPLES, len(eval_raw))))
print('Train:', len(train_raw), 'Eval:', len(eval_raw))
print(train_raw[0])


## Load Qwen3-8B and configure LoRA (BF16)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
MODEL_ID, MAX_SEQ_LENGTH = 'Qwen/Qwen3-8B', 2048
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
model.config.use_cache = False
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
print('Model and LoRA config ready.')


## Convert messages to Qwen chat template

In [ ]:
def format_example(example):
    messages = []
    for m in example['messages']:
        role, content = m.get('role', 'user'), m.get('content', '')
        if role in ('system','user','assistant') and content:
            messages.append({'role':role, 'content':content})
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=False)}
train_ds = train_raw.map(format_example, remove_columns=train_raw.column_names)
eval_ds = eval_raw.map(format_example, remove_columns=eval_raw.column_names)
print(train_ds[0]['text'][:800])


## SFT configuration and training

In [ ]:
from trl import SFTTrainer, SFTConfig
OUTPUT_DIR = '/content/qwen3-8b-chatbot-lora'
args = SFTConfig(output_dir=OUTPUT_DIR, num_train_epochs=1, per_device_train_batch_size=2, per_device_eval_batch_size=2, gradient_accumulation_steps=8, learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type='cosine', logging_steps=10, eval_strategy='steps', eval_steps=100, save_strategy='steps', save_steps=100, save_total_limit=2, bf16=True, fp16=False, gradient_checkpointing=True, max_length=MAX_SEQ_LENGTH, packing=False, dataset_text_field='text', report_to='none', optim='adamw_torch')
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds, peft_config=lora_config, processing_class=tokenizer)
trainer.model.print_trainable_parameters()
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(train_result)


## Save adapter to Google Drive

In [ ]:
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/qwen3-8b-chatbot-lora'
os.makedirs(DRIVE_DIR, exist_ok=True)
for name in os.listdir(OUTPUT_DIR):
    src, dst = os.path.join(OUTPUT_DIR,name), os.path.join(DRIVE_DIR,name)
    if os.path.isfile(src): shutil.copy2(src,dst)
    elif os.path.isdir(src): shutil.copytree(src,dst,dirs_exist_ok=True)
print('Saved:', DRIVE_DIR)


## Chat inference

In [ ]:
model = trainer.model
model.eval()
def chat(prompt, max_new_tokens=256):
    msgs = [{'role':'system','content':'Sen yardımcı, doğru ve Türkçe cevap veren bir asistansın. Bilmediğin şeyi uydurma.'},{'role':'user','content':prompt}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(chat('LoRA fine-tuning nedir? Türkçe açıkla.'))


## Troubleshooting
- CUDA OOM: MAX_SEQ_LENGTH=1024 ve batch size=1 yap.
- HF access denied: model lisans/erişim şartlarını kabul et ve Colab Secrets HF_TOKEN ekle.
- UltraChat genel sohbet dataset'idir; Türkçe/alan özelinde verilerle zenginleştir.
- Colab runtime kesilebilir; checkpoint'leri Drive'a düzenli kopyala.
- Bu notebook adapter kaydeder; full model merge/deployment içermez.
